# Notebook 02 - Feature Engineering

Ce notebook construit les variables temporelles necessaires a la classification de panne et a l'estimation de la RUL.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import prepare_raw_datasets
from src.feature_engineering import build_feature_table, temporal_train_test_split, get_feature_columns
from src.project_pipeline import run_full_training
from src.models import train_classifiers, train_rul_models
from src.evaluation import evaluate_classifier, evaluate_regressor
from src.utils import COLOR_PALETTE, RANDOM_STATE, SENSOR_COLUMNS, CRITICAL_RUL_THRESHOLD

plt.style.use("seaborn-v0_8")
sns.set_palette([COLOR_PALETTE["accent"], COLOR_PALETTE["success"], COLOR_PALETTE["info"], COLOR_PALETTE["danger"]])
pd.set_option("display.max_columns", 120)


## 1. Construction de la table de features

Les rolling windows, lags, differences et compteurs metier sont generes centralement dans `src.feature_engineering`.


In [ ]:
bundle = prepare_raw_datasets(PROJECT_ROOT)
features = build_feature_table(bundle, PROJECT_ROOT)
train_df, test_df = temporal_train_test_split(features)

# ... Nettoyage : prints supprimés ...
features.filter(regex="rolling|lag|diff|rul|failure_within_24h").head()


## 2. Verifications anti-fuite

Le split est strictement chronologique par machine : les 80% premiers en train, les 20% derniers en test. Aucun echantillon futur n'alimente l'apprentissage du passe.


In [ ]:
leakage_checks = []
for machine_id, group in features.groupby("machineID"):
    train_part = train_df[train_df["machineID"] == machine_id]
    test_part = test_df[test_df["machineID"] == machine_id]
    leakage_checks.append(
        {
            "machineID": machine_id,
            "train_max": train_part["datetime"].max(),
            "test_min": test_part["datetime"].min(),
            "is_valid": train_part["datetime"].max() <= test_part["datetime"].min(),
        }
    )
leakage_df = pd.DataFrame(leakage_checks)
leakage_df["is_valid"].value_counts()


## 3. Apercu des variables creees

On visualise les nouvelles familles de variables pour documenter le pipeline.


In [ ]:
engineered_columns = [col for col in features.columns if any(token in col for token in ["rolling", "lag", "diff", "zscore"])]
# ... Nettoyage : prints supprimés ...
pd.Series(engineered_columns[:40])


In [ ]:
sample_machine = features[features["machineID"] == 1].tail(72)
plt.figure(figsize=(15, 5))
plt.plot(sample_machine["datetime"], sample_machine["vibration"], label="Vibration brute", color=COLOR_PALETTE["danger"])
plt.plot(sample_machine["datetime"], sample_machine["vibration_rolling_mean_24h"], label="Rolling mean 24h", color=COLOR_PALETTE["success"])
plt.title("Machine 1 - Exemple de smoothing temporel")
plt.xlabel("Temps")
plt.ylabel("Vibration")
plt.legend()
plt.tight_layout()
plt.show()


## 4. Cible RUL

La Remaining Useful Life est calculee comme le nombre d'heures restantes avant la prochaine panne connue.


In [ ]:
features[["machineID", "datetime", "rul_hours", "hours_since_last_failure", "next_failure_type"]].dropna().head(10)


## 5. Export

La table finale est enregistree dans `data/processed/feature_table.csv` pour etre reutilisee par les notebooks suivants et le dashboard.
